# 12 · Model D: self-supervised pre-training on unlabelled traces

*Level: advanced. Guide Section 16. Code: `deep.Pretrainer`, `deep.PretrainedCNN`, `deep.PretrainedHead`.*

**Idea.** Labels need images and are expensive; traces are cheap. A network first learns "what traces look like" from traces alone, using
tasks that need no labels:
* fill in hidden parts of the curves;
* predict the settings;
* recognize two noisy copies of the same curves.

Then a few labelled packets suffice.

In [1]:
from tutorial_utils import *

import torch
from ml import deep
from ml.benchmark import THETA_CURVES, inputs_for

## 1. The encoder and the masks

The **encoder** $g_\omega$ is the CNN body of notebook 04, ending in 16 numbers: the **embedding** $\mathbf z$. In **masked reconstruction**,
part of the input is hidden and a decoder must rebuild it. The hidden part is either:
* blocks of 32 positions; or
* one whole curve.

Whole curves matter: short gaps can be filled by interpolation, but rebuilding the noise spectrum from the other curves forces the encoder to
learn how the curves are related.

In [2]:
df, curves, Y, groups = toy_scan(30, seed=1)
Xc, _ = inputs_for(df, curves, ("theta",), True, THETA_CURVES, log_frequency_spectra=True)
c = Xc["curves"][0]                                  # (7 curves, 256 points)
rng = np.random.default_rng(0)
block = np.repeat(rng.random(8) < 0.25, 32)[None, :].repeat(7, 0)
whole = np.zeros_like(block); whole[3] = True
fig, ax = plt.subplots(1, 3, figsize=(15, 3))
for a, mask, title in zip(ax, (np.zeros_like(block), block, whole), ("the 7 curves (standardized)", "blocks of 32 hidden", "one whole curve hidden")):
    a.imshow(np.where(mask, np.nan, (c - c.mean(1, keepdims=True)) / (c.std(1, keepdims=True) + 1e-9)), aspect="auto", cmap="viridis"); a.set_title(title)
    a.set_yticks(range(7)); a.set_yticklabels([n[:22] for n in THETA_CURVES], fontsize=7)
plt.tight_layout(); plt.show()

## 2. Contrastive learning (InfoNCE), by hand

Two noisy views of each example. For view $i$ with partner $i^+$:

$$\ell_i = -\log\frac{\exp(\mathrm{sim}(\mathbf v_i,\mathbf v_{i^+})/\tau)}{\sum_{k\ne i}\exp(\mathrm{sim}(\mathbf v_i,\mathbf v_k)/\tau)},\qquad \mathrm{sim} = \text{cosine similarity}.$$

* The loss is small when each view's partner is the most similar of all the other views.
* τ (the temperature) sharpens the contrast.

In [3]:
emb = rng.normal(size=(4, 8)); views = np.concatenate([emb + 0.1 * rng.normal(size=emb.shape), emb + 0.1 * rng.normal(size=emb.shape)])
v = views / np.linalg.norm(views, axis=1, keepdims=True); sim = v @ v.T
partner = np.r_[np.arange(4, 8), np.arange(0, 4)]
for tau in (1.0, 0.2, 0.05):
    logits = sim / tau - 1e9 * np.eye(8)
    loss = np.mean([-(logits[i, partner[i]] - np.log(np.exp(logits[i]).sum())) for i in range(8)])
    print(f"τ = {tau}: InfoNCE loss {loss:.3f}")
shuffled = np.r_[np.arange(1, 8), 0]
logits = sim / 0.2 - 1e9 * np.eye(8)
print("with WRONG partners the loss is much larger:", np.mean([-(logits[i, shuffled[i]] - np.log(np.exp(logits[i]).sum())) for i in range(8)]).round(3))

τ = 1.0: InfoNCE loss 1.183
τ = 0.2: InfoNCE loss 0.071
τ = 0.05: InfoNCE loss 0.000
with WRONG partners the loss is much larger: 5.598


## 3. Pre-training on 400 trace-only packets (1600 traces, no labels used)

The total loss is $L = L_\text{rec} + L_\theta + 0.3\,L_\text{con}$ (reconstruction, settings prediction, contrastive).

In [4]:
du, cu, _, _ = toy_scan(400, seed=7)
Xu, _ = inputs_for(du, cu, ("theta",), True, THETA_CURVES, log_frequency_spectra=True)
pre = deep.Pretrainer(epochs=20).fit(Xu["curves"], du[data.columns(du, "theta")].to_numpy(float))
fig, ax = plt.subplots(1, 3, figsize=(13, 2.8))
for a, (k, v_) in zip(ax, pre.losses.items()): a.plot(v_); a.set_title(k); a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

**What did it learn?** A linear "probe" predicts hidden quantities of the toy world from the 16-number embedding. The comparison is
against an untrained (random) encoder.

In [5]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score
z_pre = pre.encode(Xu["curves"])
rand = deep.Pretrainer(epochs=0).fit(Xu["curves"][:10]); z_rand = rand.encode(Xu["curves"])
for q in ("truth.floor", "truth.sigma1", "truth.state"):
    t = du[q].to_numpy()
    r2 = lambda z: cross_val_score(RidgeCV(), z, t, cv=5, scoring="r2").mean()
    print(f"{q:14s}: probe R² from the pre-trained embedding {r2(z_pre):5.2f}   from a random encoder {r2(z_rand):5.2f}")

truth.floor   : probe R² from the pre-trained embedding  0.33   from a random encoder  0.08
truth.sigma1  : probe R² from the pre-trained embedding  0.98   from a random encoder  0.82
truth.state   : probe R² from the pre-trained embedding  0.24   from a random encoder  0.31


## 4. With few labels: from scratch against pre-trained

There are two ways to use the encoder:
* **fine-tuning** (`PretrainedCNN`): start the CNN from the pre-trained weights and train everything with a small learning rate;
* **frozen** (`PretrainedHead`): keep the encoder fixed and train a small network on the embedding.

Below, fine-tuning is compared with a CNN trained from scratch.

In [6]:
dft, ct, Yt, _ = toy_scan(60, seed=2)
Xt, _ = inputs_for(dft, ct, ("theta", "ref"), True, THETA_CURVES, log_frequency_spectra=True)   # settings + own reference features
for n in (30, 60):
    dtr, ctr, Ytr, _ = toy_scan(n, seed=1)
    Xtr, _ = inputs_for(dtr, ctr, ("theta", "ref"), True, THETA_CURVES, log_frequency_spectra=True)
    m, s = models.CurveCNN(n_models=2).fit(Xtr, Ytr).predict(Xt); _ = report(f"CNN from scratch, {n} labelled packets", Yt, m, s)
    m, s = deep.PretrainedCNN(pre, n_models=2).fit(Xtr, Ytr).predict(Xt); _ = report(f"pre-trained + fine-tuned, {n} packets", Yt, m, s)

CNN from scratch, 30 labelled packets    sigma_e:   7.5 % (cover 0.70)   lambda:  45.8 % (cover 0.08)   gain:   1.9 % (cover 0.89)


pre-trained + fine-tuned, 30 packets     sigma_e:   9.7 % (cover 0.76)   lambda:  52.7 % (cover 0.07)   gain:   6.4 % (cover 0.50)


CNN from scratch, 60 labelled packets    sigma_e:   4.3 % (cover 0.97)   lambda:   8.6 % (cover 0.81)   gain:   1.5 % (cover 0.76)


pre-trained + fine-tuned, 60 packets     sigma_e:   4.3 % (cover 0.82)   lambda:  15.2 % (cover 0.83)   gain:   1.9 % (cover 0.84)


**Reading the numbers:**
* The probe shows what pre-training bought: the embedding knows the single-skip noise and, unlike a random encoder, part of the noise
  floor, without seeing a single label.
* On the labels, pre-training did **not** help in this run (σe 9.7 % against 7.5 % from scratch at 30 packets; equal at 60). One training
  scan is noisy: over 5 scans in the project's test, pre-training helped at 60 packets (6.6 against 8.9 %) and no longer at 240. The
  benefit is small and not guaranteed. On real data it has to earn its place against the CNN trained from scratch.

## When to use it

* **Good:** many unlabelled traces and few labelled packets (it helped most at 30–120 packets in our tests).
* **Less useful:** with many labels, a CNN from scratch catches up.

The pre-training tasks must make the encoder notice what matters. It only learned the noise floor once the spectra were on a log-frequency
axis.

Next: **13 · Deep kernel learning (C).**